# FOMC Rate Decisions and Market Reaction: Event Study

## What I'm trying to find out

I want to check if the stock market actually reacts to Fed rate decisions,
and if it does, where that reaction shows up - in the market as a whole, or
in individual stocks. I split this into 3 questions, each one building on
the last:

1. Does SPY (the market) get more volatile around FOMC days than a normal day?
2. Do individual stocks (AAPL, TLT, XLF) just move with the market on these
   days, or do some of them move *more* than what their usual relationship
   with the market would predict?
3. (extra / exploratory) Does VIX go up before the announcement and drop
   after, like it's supposed to?




In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import wilcoxon, mannwhitneyu
import statsmodels.api as sm
from statsmodels.stats.stattools import jarque_bera, durbin_watson
from statsmodels.stats.diagnostic import het_arch

%matplotlib inline
RNG = np.random.default_rng(42)

## 1. Getting the data and building the event list


In [6]:
def load_price_series(path, name):
    raw = pd.read_csv(path, skiprows=[1, 2]).rename(columns={"Price": "Date"})
    raw["Date"] = pd.to_datetime(raw["Date"])
    raw = raw.sort_values("Date").reset_index(drop=True)
    raw[f"{name}_ret"] = raw["Close"].pct_change()
    return raw[["Date", f"{name}_ret"]]

spy = load_price_series("spy_2013_2026.csv", "spy").rename(columns={"spy_ret": "ret"})
spy["abs_ret"] = spy["ret"].abs()
spy = spy.dropna(subset=["ret"]).reset_index(drop=True)

aapl = load_price_series("aapl_2013_2026.csv", "aapl")
tlt = load_price_series("tlt_2013_2026.csv", "tlt")
xlf = load_price_series("xlf_2013_2026.csv", "xlf")

trading_dates = sorted(spy["Date"].tolist())
trading_dates_set = set(trading_dates)

print(f"SPY: {len(spy)} trading days ({spy['Date'].min().date()} to {spy['Date'].max().date()})")

SPY: 3430 trading days (2013-01-03 to 2026-08-24)


In [7]:
# Upload fomc events
fomc_events = pd.read_csv("fomc_events.csv")
fomc_events["announcement_date"] = pd.to_datetime(fomc_events["announcement_date"])
fomc_events = fomc_events[
    (fomc_events["announcement_date"] >= spy["Date"].min()) &
    (fomc_events["announcement_date"] <= spy["Date"].max())
].reset_index(drop=True)

def map_to_event_day(ann_date, timing, trading_dates_sorted):
    candidates = ([d for d in trading_dates_sorted if d > ann_date] if timing == "after_hours"
                  else [d for d in trading_dates_sorted if d >= ann_date])
    return candidates[0] if candidates else None

fomc_events["event_trading_day"] = fomc_events.apply(
    lambda r: map_to_event_day(r["announcement_date"], r["timing"], trading_dates), axis=1)

WINDOW = 1  # [-1, 0, +1]
event_windows = {}
for _, row in fomc_events.iterrows():
    ed = row["event_trading_day"]
    if ed not in trading_dates_set:
        continue
    idx = trading_dates.index(ed)
    lo, hi = idx - WINDOW, idx + WINDOW
    if 0 <= lo and hi < len(trading_dates):
        event_windows[ed] = list(range(lo, hi + 1))

print(f"Events in sample: {len(fomc_events)} | Event windows built: {len(event_windows)}")
fomc_events[["announcement_date", "direction", "scheduled", "event_trading_day"]]

Events in sample: 31 | Event windows built: 31


,announcement_date,direction,scheduled,event_trading_day
0,2015-12-16,hike,True,2015-12-16
1,2016-12-14,hike,True,2016-12-14
2,2017-03-15,hike,True,2017-03-15
3,2017-06-14,hike,True,2017-06-14
4,2017-12-13,hike,True,2017-12-13
5,2018-03-21,hike,True,2018-03-21
6,2018-06-13,hike,True,2018-06-13
7,2018-09-26,hike,True,2018-09-26
8,2018-12-19,hike,True,2018-12-19
9,2019-07-31,cut,True,2019-07-31


## 2. Question 1 - Is the market more volatile around FOMC days?

**Method:** for each event, take the 3-day window `[-1,0,+1]` and turn it
into one number (average |daily return|), then compare that against normal
3-day windows built from days that aren't near any event. I run 4 different
tests (Welch's t-test, Mann-Whitney U, a permutation test, and a bootstrap
CI) instead of just one, so I'm not relying on a single test that could be
misleading.


In [8]:
event_level_rows = []
for ed, idx_list in event_windows.items():
    w = spy.iloc[idx_list]
    event_level_rows.append({"event_date": ed, "avg_ret": w["ret"].mean(), "avg_abs_ret": w["abs_ret"].mean()})
event_level_df = pd.DataFrame(event_level_rows).merge(
    fomc_events[["event_trading_day", "direction", "scheduled"]],
    left_on="event_date", right_on="event_trading_day", how="left"
).drop(columns="event_trading_day").sort_values("event_date").reset_index(drop=True)

BUFFER = 2
excluded_idx = set()
for ed in event_windows:
    idx = trading_dates.index(ed)
    for off in range(-WINDOW - BUFFER, WINDOW + BUFFER + 1):
        i = idx + off
        if 0 <= i < len(trading_dates):
            excluded_idx.add(i)
eligible_idx = sorted(i for i in range(len(spy)) if i not in excluded_idx)

non_event_stats = []
i = 0
while i + 2 < len(eligible_idx):
    if eligible_idx[i + 2] - eligible_idx[i] == 2:
        r = spy.iloc[eligible_idx[i]: eligible_idx[i] + 3]
        non_event_stats.append({"avg_ret": r["ret"].mean(), "avg_abs_ret": r["abs_ret"].mean()})
        i += 3
    else:
        i += 1
non_event_df = pd.DataFrame(non_event_stats)

print(f"Event 3-day windows: {len(event_level_df)} | Non-event 3-day windows: {len(non_event_df)}")

Event 3-day windows: 31 | Non-event 3-day windows: 1061


In [9]:
def run_tests(x, y, label, n_iter=10_000, rng=RNG):
    t_stat, p_welch = stats.ttest_ind(x, y, equal_var=False)
    u_stat, p_mw = mannwhitneyu(x, y, alternative="two-sided")

    observed = x.mean() - y.mean()
    combined = np.concatenate([x, y]); n_x = len(x)
    perm_diffs = np.empty(n_iter)
    for k in range(n_iter):
        rng.shuffle(combined)
        perm_diffs[k] = combined[:n_x].mean() - combined[n_x:].mean()
    p_perm = np.mean(np.abs(perm_diffs) >= np.abs(observed))

    boot_diffs = np.array([rng.choice(x, len(x), replace=True).mean() - rng.choice(y, len(y), replace=True).mean()
                            for _ in range(n_iter)])
    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])

    print(f"--- {label} ---")
    print(f"  event mean={x.mean():.5f}  non-event mean={y.mean():.5f}  diff={observed:.5f}")
    print(f"  Welch t-test:   p={p_welch:.4f}   Mann-Whitney: p={p_mw:.4f}")
    print(f"  Permutation:    p={p_perm:.4f}   Bootstrap 95% CI: [{ci_low:.5f}, {ci_high:.5f}]")
    print()

print("=== Direction (mean return) - is there a systematic drift? ===")
run_tests(event_level_df["avg_ret"].values, non_event_df["avg_ret"].values, "Mean return")

print("=== Volatility (|return|) - is there elevated volatility? ===")
run_tests(event_level_df["avg_abs_ret"].values, non_event_df["avg_abs_ret"].values, "Full sample (31 events)")

# note: this only drops the 2 emergency-cut events (+ their +-2 day buffer).
# it doesn't remove COVID volatility from the rest of the "normal" days, so
# it's a smaller check than a full COVID-robustness test.
emergency_mask = ~event_level_df["scheduled"]
run_tests(event_level_df.loc[~emergency_mask, "avg_abs_ret"].values, non_event_df["avg_abs_ret"].values,
          "Excluding the 2 emergency rate-cut events (Mar 2020) from the event sample")

=== Direction (mean return) - is there a systematic drift? ===
--- Mean return ---
  event mean=0.00015  non-event mean=0.00080  diff=-0.00065
  Welch t-test:   p=0.6742   Mann-Whitney: p=0.2161
  Permutation:    p=0.4944   Bootstrap 95% CI: [-0.00361, 0.00236]

=== Volatility (|return|) - is there elevated volatility? ===
--- Full sample (31 events) ---
  event mean=0.01216  non-event mean=0.00667  diff=0.00548
  Welch t-test:   p=0.0550   Mann-Whitney: p=0.0205
  Permutation:    p=0.0000   Bootstrap 95% CI: [0.00120, 0.01162]

--- Excluding the 2 emergency rate-cut events (Mar 2020) from the event sample ---
  event mean=0.00882  non-event mean=0.00667  diff=0.00215
  Welch t-test:   p=0.0633   Mann-Whitney: p=0.0777
  Permutation:    p=0.0221   Bootstrap 95% CI: [0.00004, 0.00434]



### Answer to Q1



*   **Mean return:** No evidence that FOMC days consistently push the market up or down.
*   **Volatility:** FOMC days were more volatile than non-FOMC days (1.22% vs. 0.67%).
*   **Excluding March 2020 emergency cuts:** The effect became smaller (1.22% → 0.88%), but the permutation test remained significant (p = 0.022).






## 3. Question 2a - Do individual stocks move with the market?

Before I can say a stock reacts "abnormally," I first need to know if it
even moves in the same direction as the market on these days. I added TLT
(long treasuries, tied directly to rate expectations) and XLF (financial
sector) alongside AAPL.

**Method:** for each event's 3-day window, compute each asset's return,
compare its sign to SPY's, and check the correlation across all 31 events.


In [10]:
df_all = spy[["Date", "ret"]].rename(columns={"ret": "spy_ret"}).merge(aapl, on="Date").merge(tlt, on="Date").merge(xlf, on="Date").dropna().reset_index(drop=True)
dates_all = sorted(df_all["Date"].tolist())

comove_rows = []
for _, row in fomc_events.iterrows():
    ed = row["event_trading_day"]
    if ed not in dates_all:
        continue
    idx = dates_all.index(ed)
    if idx - 1 < 0 or idx + 1 >= len(dates_all):
        continue
    w = df_all.iloc[idx - 1: idx + 2]
    # compounding properly: (1+R1)(1+R2)(1+R3) - 1, not just adding the 3
    # daily returns up (adding is only an approximation).
    comove_rows.append({
        "event_date": ed.date(), "direction": row["direction"],
        "SPY_%": ((1 + w["spy_ret"]).prod() - 1) * 100,
        "AAPL_%": ((1 + w["aapl_ret"]).prod() - 1) * 100,
        "TLT_%": ((1 + w["tlt_ret"]).prod() - 1) * 100,
        "XLF_%": ((1 + w["xlf_ret"]).prod() - 1) * 100,
    })
comove_df = pd.DataFrame(comove_rows)

print("=== % of events moving the same direction as SPY, and correlation ===")
for asset in ["AAPL", "TLT", "XLF"]:
    same_dir = (np.sign(comove_df["SPY_%"]) == np.sign(comove_df[f"{asset}_%"])).mean() * 100
    corr = comove_df["SPY_%"].corr(comove_df[f"{asset}_%"])
    print(f"{asset}: {same_dir:.0f}% same direction | correlation r = {corr:.3f}")

comove_df.round(2)

=== % of events moving the same direction as SPY, and correlation ===
AAPL: 74% same direction | correlation r = 0.826
TLT: 52% same direction | correlation r = -0.070
XLF: 84% same direction | correlation r = 0.901


,event_date,direction,SPY_%,AAPL_%,TLT_%,XLF_%
0,2015-12-16,hike,0.97,-3.11,0.32,2.51
1,2016-12-14,hike,0.25,2.22,-0.26,0.64
2,2017-03-15,hike,0.28,1.07,1.19,-0.28
3,2017-06-14,hike,0.17,-0.78,1.40,0.29
4,2017-12-13,hike,-0.24,-0.26,1.14,-0.89
5,2018-03-21,hike,-2.52,-3.68,0.69,-3.49
6,2018-06-13,hike,0.06,-0.22,0.80,-1.53
7,2018-09-26,hike,-0.11,1.88,0.67,-1.94
8,2018-12-19,hike,-3.21,-4.34,1.56,-2.50
9,2019-07-31,cut,-2.20,-0.60,3.06,-3.02


### Answer to Q2a

- **AAPL and XLF move with the market most of the time** (74% and 84%
  same direction, r=0.83 and r=0.90) - that's just beta, they're stocks so
  they tend to follow the market.
- **TLT doesn't** (52% same direction, r about -0.07, basically random).
  Makes sense - bonds respond to rate expectations directly, not to
  whatever mood equities are in that day.


## 3b. Question 2b - Do any of them react beyond just following the market?

Q2a mixes two things together: "this stock follows the market anyway" and
"this stock is specifically reacting to the Fed." The Market Model
separates them: estimate each stock's normal alpha/beta vs SPY from a clean
window before the event (with a 21-day gap so the estimate isn't already
picking up anticipation), then check the **Cumulative Abnormal Return
(CAR)** - the part of the move that beta alone doesn't explain.


In [11]:
def market_model_car(asset_ret_df, ret_col, label, est_len=120, gap=21, window=1):
    merged = pd.merge(asset_ret_df, spy[["Date", "ret"]].rename(columns={"ret": "mkt_ret"}), on="Date").dropna().reset_index(drop=True)
    dates_list = merged["Date"].tolist()

    def estimate(event_date):
        if event_date not in dates_list:
            return None
        idx = dates_list.index(event_date)
        est_start, est_end = idx - gap - est_len, idx - gap
        if est_start < 0:
            return None
        est_data = merged.iloc[est_start:est_end]
        X = sm.add_constant(est_data["mkt_ret"])
        model = sm.OLS(est_data[ret_col], X).fit()
        alpha, beta = model.params["const"], model.params["mkt_ret"]
        event_rows = merged.iloc[max(0, idx - window): idx + window + 1]
        expected = alpha + beta * event_rows["mkt_ret"]
        ar = event_rows[ret_col] - expected
        return {"car": ar.sum(), "beta": beta, "residuals": model.resid}

    results = {}
    for ed in fomc_events["event_trading_day"]:
        r = estimate(ed)
        if r is not None:
            results[ed] = r

    cars = np.array([r["car"] for r in results.values()])
    betas = np.array([r["beta"] for r in results.values()])
    t_stat, p_val = stats.ttest_1samp(cars, 0)
    w_stat, w_p = wilcoxon(cars)
    boot = np.array([RNG.choice(cars, len(cars), replace=True).mean() for _ in range(10_000)])
    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

    print(f"--- {label} ---")
    print(f"  events: {len(cars)}  mean beta: {betas.mean():.2f}  mean CAR: {cars.mean():+.5f}")
    print(f"  t-test: p={p_val:.4f}  Wilcoxon: p={w_p:.4f}  bootstrap 95% CI: [{ci_low:.5f}, {ci_high:.5f}]")
    print()
    return results

aapl_results = market_model_car(aapl, "aapl_ret", "AAPL")
tlt_results  = market_model_car(tlt,  "tlt_ret",  "TLT")
xlf_results  = market_model_car(xlf,  "xlf_ret",  "XLF")

--- AAPL ---
  events: 31  mean beta: 1.24  mean CAR: +0.00072
  t-test: p=0.8450  Wilcoxon: p=0.7498  bootstrap 95% CI: [-0.00651, 0.00760]

--- TLT ---
  events: 31  mean beta: -0.14  mean CAR: +0.00442
  t-test: p=0.1166  Wilcoxon: p=0.1066  bootstrap 95% CI: [-0.00100, 0.00957]

--- XLF ---
  events: 31  mean beta: 0.98  mean CAR: -0.00248
  t-test: p=0.3070  Wilcoxon: p=0.1887  bootstrap 95% CI: [-0.00697, 0.00238]



In [12]:
# checking residuals per window instead of dumping them all together -
# pooling would mix calm years with crazy years (like 2022) and could make
# it look like there's more ARCH/non-normality going on than there really is.
def diagnostics_summary(results, label):
    rows = []
    for d, r in results.items():
        resid = r["residuals"].values
        if len(resid) < 20:
            continue
        _, jb_p, _, _ = jarque_bera(resid)
        try:
            _, arch_p, _, _ = het_arch(resid)
        except ValueError:
            arch_p = np.nan
        rows.append({"jb_p": jb_p, "arch_p": arch_p, "dw": durbin_watson(resid)})
    diag = pd.DataFrame(rows)
    print(f"{label}: {(diag['jb_p']<0.05).mean():.0%} windows non-normal | "
          f"{(diag['arch_p']<0.05).mean():.0%} windows show ARCH | mean DW={diag['dw'].mean():.2f}")

diagnostics_summary(aapl_results, "AAPL")
diagnostics_summary(tlt_results, "TLT")
diagnostics_summary(xlf_results, "XLF")

AAPL: 87% windows non-normal | 10% windows show ARCH | mean DW=1.84
TLT: 10% windows non-normal | 10% windows show ARCH | mean DW=2.01
XLF: 52% windows non-normal | 16% windows show ARCH | mean DW=1.97


C:\Users\admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\stats\diagnostic.py:997: FutureWarning: acorr_lm currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return acorr_lm(




### Q2a — Do they move with the market?

**AAPL and XLF generally move with the market**, while **TLT does not**. AAPL and XLF have high correlations with SPY (0.83 and 0.90), whereas TLT is almost uncorrelated (−0.07).

### Q2b — Do any of them react beyond just following the market?

**None of the three shows a significant abnormal return.** TLT has the closest result (p ≈ 0.12), but the evidence is still too weak to claim a clear FOMC-specific effect.


## 4. Question 3 (extra) - Does VIX behave the way it's supposed to?

If FOMC decisions actually resolve uncertainty, VIX should go up before
the meeting (people get nervous) and drop after (uncertainty's gone -
"vol crush").


In [13]:
vix_raw = pd.read_csv("vix_2013_2026.csv", skiprows=[1, 2]).rename(columns={"Price": "Date"})
vix_raw["Date"] = pd.to_datetime(vix_raw["Date"])
vix_lookup = dict(zip(vix_raw["Date"], vix_raw["Close"]))

PRE, POST = 3, 3
vix_rows = []
for ed in event_windows:
    idx = trading_dates.index(ed)
    if idx - PRE < 0 or idx + POST >= len(trading_dates):
        continue
    pre_day, post_day = trading_dates[idx - PRE], trading_dates[idx + POST]
    if ed in vix_lookup and pre_day in vix_lookup and post_day in vix_lookup:
        vix_rows.append({
            "event_date": ed,
            "run_up": vix_lookup[ed] - vix_lookup[pre_day],
            "crush": vix_lookup[post_day] - vix_lookup[ed],
        })
vix_df = pd.DataFrame(vix_rows)

t_up, p_up = stats.ttest_1samp(vix_df["run_up"], 0)
t_down, p_down = stats.ttest_1samp(vix_df["crush"], 0)

print(f"Events measured: {len(vix_df)}")
print(f"Mean VIX change, {PRE}d BEFORE event: {vix_df['run_up'].mean():+.3f}  (t-test p={p_up:.4f})")
print(f"Mean VIX change, {POST}d AFTER event:  {vix_df['crush'].mean():+.3f}  (t-test p={p_down:.4f})")

Events measured: 31
Mean VIX change, 3d BEFORE event: +1.054  (t-test p=0.3671)
Mean VIX change, 3d AFTER event:  +0.378  (t-test p=0.6369)


### Q3 — VIX reaction

**VIX rises slightly before FOMC days**, which is consistent with market expectations. However, there is no clear drop after the event. This may be because daily data is too coarse to capture the **“vol crush”** that usually happens within hours around the announcement.
